In [0]:
from pyspark.sql import functions as F

# Azure Storage location
STORAGE_ROOT = (
    "abfss://lakehouse@storagelokiiiiiiii.dfs.core.windows.net"
)

TABLES = [
    "customers",
    "products",
    "orders",
    "employees",
    "departments"
]

for table_name in TABLES:

    print(f"Starting ingestion for {table_name}")

    # Input folder for this dataset
    source_path = (
        f"{STORAGE_ROOT}/landing/{table_name}"
    )

    # Each dataset needs its own schema location
    schema_path = (
        f"{STORAGE_ROOT}/system/autoloader/schemas/{table_name}"
    )

    # Each dataset needs its own checkpoint
    checkpoint_path = (
        f"{STORAGE_ROOT}/system/autoloader/checkpoints/{table_name}"
    )

    source_df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "rescue")
        .option("rescuedDataColumn", "_rescued_data")
        .option("header", "true")
        .load(source_path)
    )

    # Convert the watermark field into a timestamp
    source_df = source_df.withColumn(
        "last_updated",
        F.to_timestamp(F.col("last_updated"))
    )

    query = (
        source_df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(
            f"source_catalog.raw.{table_name}"
        )
    )

    # Wait until this dataset's available files are processed
    query.awaitTermination()

    print(f"Completed ingestion for {table_name}")

print("All source ingestion completed.")